In [0]:
%load_ext autoreload
%autoreload 2

import sys
import os
sys.path.append(os.getcwd())

# Imports dos Módulos do Projeto
from src.ingestion.config import Config
# Configurações de Otimização do Spark (Delta Lake)
spark.conf.set("spark.databricks.delta.optimizeWrite.enabled", "true")
spark.conf.set("spark.databricks.delta.autoCompact.enabled", "true")

In [0]:
config = Config(spark)

In [0]:
config.CATALOG

## Chronos:

In [0]:
spark.sql(f'''
SELECT data, forecast, upper, lower, data_reference, 'Lojas' as canal 
FROM {config.CATALOG}.cvc.previsao_lojas_futuro_global
''')

## Individual Loja:

In [0]:
spark.sql(
f'''
CREATE OR REPLACE TABLE {config.CATALOG}.cvc.previsao_venda_todos_canais_fotografia AS
WITH Metricas_Consolidadas AS (
    -- Unificamos as métricas de erro de todos os canais primeiro
    SELECT codigo_loja, versao, metrica_mes, previsao, real, 'Venda Direta' AS canal FROM {config.CATALOG}.cvc.resultado_metricas_treinamento_venda_direta
    UNION ALL
    SELECT codigo_loja, versao, metrica_mes, previsao, real, 'Ecommerce' AS canal FROM {config.CATALOG}.cvc.resultado_metricas_treinamento_ecommerce
    UNION ALL
    SELECT codigo_loja, versao, metrica_mes, previsao, real, 'Lojas' AS canal FROM {config.CATALOG}.cvc.resultado_metricas_treinamento_lojas
),
Metricas_Agrupadas AS (
    -- Calculamos o RMSE por canal/loja e versão
    SELECT
        canal,
        codigo_loja,
        versao,
        SQRT(AVG(POWER(previsao - real, 2))) AS rmse
    FROM Metricas_Consolidadas
    WHERE previsao IS NOT NULL AND real IS NOT NULL
    GROUP BY canal, codigo_loja, versao
),
RMSE_Final AS (
    -- Filtramos a última versão de métrica por loja dentro de cada canal
    SELECT * FROM Metricas_Agrupadas
    QUALIFY ROW_NUMBER() OVER (PARTITION BY canal, codigo_loja ORDER BY versao DESC) = 1
),
Previsoes_Consolidadas AS (
    -- Unificamos as previsões de todos os canais
    SELECT codigo_loja, version_pipeline, data_previsao, previsao_venda, data_reference, 'Venda Direta' AS canal FROM {config.CATALOG}.cvc.previsao_venda_direta_futuro
    UNION ALL
    SELECT codigo_loja, version_pipeline, data_previsao, previsao_venda, data_reference, 'Ecommerce' AS canal FROM {config.CATALOG}.cvc.previsao_ecommerce_futuro
    UNION ALL
    SELECT codigo_loja, version_pipeline, data_previsao, previsao_venda, data_reference, 'Lojas' AS canal FROM {config.CATALOG}.cvc.previsao_lojas_futuro
),
Previsoes_Finais AS (
    -- Identificamos a última data_reference sem descartar os múltiplos meses (data_previsao)
    SELECT 
        canal,
        codigo_loja,
        version_pipeline,
        data_previsao,
        previsao_venda AS valor_previsao,
        data_reference,
        -- 't' calculado por data_previsao dentro da carga atual
        ROW_NUMBER() OVER (PARTITION BY canal, codigo_loja, data_reference, version_pipeline ORDER BY data_previsao) AS t,
        -- Marca qual é a carga mais recente para cada loja/canal
        MAX(data_reference) OVER (PARTITION BY canal, codigo_loja) AS ultima_data_ref
    FROM Previsoes_Consolidadas
)
-- JOIN FINAL
SELECT 
    p.canal,
    p.codigo_loja,
    p.data_previsao,
    p.version_pipeline,
    p.valor_previsao AS previsao_central,
    p.t AS passo_tempo,
    ROUND(r.rmse, 2) AS rmse_base,
    ROUND(1.96 * COALESCE(r.rmse, 0) * SQRT(p.t), 2) AS margem_erro,
    ROUND(p.valor_previsao - (1.96 * COALESCE(r.rmse, 0) * SQRT(p.t)), 2) AS limite_inferior,
    ROUND(p.valor_previsao + (1.96 * COALESCE(r.rmse, 0) * SQRT(p.t)), 2) AS limite_superior
FROM Previsoes_Finais p
LEFT JOIN RMSE_Final r 
    ON p.canal = r.canal 
    AND p.codigo_loja = r.codigo_loja 
    AND p.version_pipeline = r.versao
WHERE p.data_reference = p.ultima_data_ref -- Aqui garantimos que pegamos todos os meses da última carga
ORDER BY p.canal, p.codigo_loja, p.data_previsao;
'''   
)

In [0]:
spark.sql(
f'''
CREATE OR REPLACE TABLE {config.CATALOG}.cvc.previsao_venda_todos_canais_global_fotografia AS

SELECT data as data_previsao, forecast, upper, lower, data_reference, 'Ecommerce' as canal 
FROM {config.CATALOG}.cvc.previsao_ecommerce_futuro_global
WHERE data = (SELECT MAX(data) FROM {config.CATALOG}.cvc.previsao_ecommerce_futuro_global)

UNION ALL

SELECT data as data_previsao, forecast, upper, lower, data_reference, 'Venda Direta' as canal 
FROM {config.CATALOG}.cvc.previsao_venda_direta_futuro_global
WHERE data_reference = (SELECT MAX(data_reference) FROM {config.CATALOG}.cvc.previsao_venda_direta_futuro_global)

UNION ALL

SELECT data as data_previsao, forecast, upper, lower, data_reference, 'Lojas' as canal 
FROM {config.CATALOG}.cvc.previsao_lojas_futuro_global
WHERE data = (SELECT MAX(data) FROM {config.CATALOG}.cvc.previsao_lojas_futuro_global)
'''
)

In [0]:
spark.sql(
f'''
CREATE OR REPLACE TABLE {config.CATALOG}.cvc.previsao_venda_reconciliada_final AS
WITH Darts_Pesos AS (
    SELECT 
        canal,
        data_previsao,
        codigo_loja,
        previsao_central,
        rmse_base as rmse_darts,
        SUM(previsao_central) OVER (PARTITION BY canal, data_previsao) as soma_canal_darts
    FROM {config.CATALOG}.cvc.previsao_venda_todos_canais_fotografia
),
Chronos_Global AS (
    SELECT 
        data_previsao,
        canal,
        forecast as previsao_chronos,
        -- Estimativa de 1 desvio padrão do Chronos (SD = (Upper-Lower)/3.92)
        (upper - lower) / 3.92 as rmse_chronos 
    FROM {config.CATALOG}.cvc.previsao_venda_todos_canais_global_fotografia
),
Calculo_Base AS (
    SELECT 
        d.canal,
        d.data_previsao,
        d.codigo_loja,
        -- VALOR CENTRAL RECONCILIADO
        CASE 
            WHEN d.soma_canal_darts > 0 
            THEN (d.previsao_central / d.soma_canal_darts) * c.previsao_chronos 
            ELSE 0 
        END AS previsao_reconciliada,
        -- COMBINAÇÃO DE ERROS (Soma dos Quadrados)
        SQRT(
          POWER(
            CASE 
              WHEN d.soma_canal_darts > 0 
              THEN (d.previsao_central / d.soma_canal_darts) * c.rmse_chronos 
              ELSE 0 
            END, 2
          ) + 
          POWER(d.rmse_darts, 2)
        ) AS rmse_combinado
    FROM Darts_Pesos d
    INNER JOIN Chronos_Global c 
        ON d.data_previsao = c.data_previsao 
        AND d.canal = c.canal
)
SELECT 
    canal,
    data_previsao,
    codigo_loja,
    -- Cálculo numérico das bandas (Z=1.96 para 95% de confiança)
    ROUND(previsao_reconciliada, 2) AS forecast,
    
    ROUND(GREATEST(0, previsao_reconciliada - (1.96 * rmse_combinado)), 2) AS lower,
    
    ROUND(previsao_reconciliada + (1.96 * rmse_combinado), 2) AS upper
FROM Calculo_Base
ORDER BY canal, data_previsao, codigo_loja;
'''
)